# Real-World Solution for SCD Type - 1 Dim

Initial Load/Full Load + Incremental Load

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text('init_load_flag', '1')

init_load_flag = int(dbutils.widgets.get('init_load_flag'))

print(init_load_flag)

# Data Reading from Silver Layer

In [0]:
df = spark.read.table('medallion_arc_e2e.silver.customers')

display(df)

# Removing Duplicates

In [0]:
df = df.dropDuplicates(subset=['customer_id'])

display(df.limit(10))

# Dividing New vs Old Records

In [0]:
if init_load_flag == 0:

    df_old = spark.sql('''select dim_customer_key, customer_id, create_date, update_date
                       from medallion_arc_e2e.gold.dim_customers''')
else:

    df_old = spark.sql('''select 0 dim_customer_key, 0 customer_id, 0 create_date, 0 update_date
                       from medallion_arc_e2e.silver.customers where 1=0''')

In [0]:
display(df_old)

# Renaming Columns of df_old

In [0]:
df_old = df_old.withColumnRenamed("dim_customer_key", "old_dim_customer_key")\
    .withColumnRenamed("customer_id", "old_customer_id")\
    .withColumnRenamed("create_date", "old_create_date")\
    .withColumnRenamed("update_date", "old_update_date")
display(df_old)

# Applying Join With the Old Records

In [0]:
df_join = df.join(df_old, df.customer_id == df_old.old_customer_id, 'left')

display(df_join)
# df = df_join.select(df.customer_id.alias('customer_id'),
#                     df.email.alias('email'),
#                     df.city.alias('city'),
#                     df.state.alias('state'),
#                     df.ingesttime.alias('ingesttime'),
#                     df.source_file.alias('source_file'),
#                     df.domains.alias('domains'),
#                     df.full_name.alias('full_name'),
#                     df_old.dim)

# Seperating New vs Old Records

In [0]:
df_new = df_join.filter(df_join.old_dim_customer_key.isNull())

In [0]:
df_old = df_join.filter(df_join.old_dim_customer_key.isNotNull())



# df_new = df_new.select(df.customer_id.alias('customer_id'),
#                        df.email.alias('email'),
#                        df.city.alias('city'),
#                        df.state.alias('state'),
#                        df.ingesttime.alias('ingesttime'),
#                        df.source_file.alias('source_file'),
#                        df.domains.alias('domains'),
#                        df.full_name.alias('full_name'),
#                        df_old.old_dim_customer_key.alias('dim_customer'))
# df_old = df_old.select(df_old.old_dim_customer_key.alias('dim_customer_key'),
#                        df_old.old_customer_id.alias('customer_id'),
#                        df_old.old_create_date.alias('create_date'),
#                        df_old.old_update_date.alias('update_date'))
# df_new = df_new.union(df_old)

In [0]:
display(df_new)

In [0]:
display(df_old)

# Preparing df_old

In [0]:
# Dropping All Columns Which are not Required

# df_old = df_old.drop('old_dim_customer_key','old_customer_id','old_update_date')
df_old = df_old.drop('old_customer_id','old_update_date')

# Renaming "old_dim_customer_key" to "dim_customer_key"
df_old = df_old.withColumnRenamed("old_dim_customer_key", "dim_customer_key")

# Renaming "old_create_date" to "create_date"
df_old = df_old.withColumnRenamed("old_create_date", "create_date")
# df_old = df_old.withColumn("create_date", to_timestamp(col('create_date')))
df_old = df_old.withColumn("create_date", col('create_date').cast('timestamp'))

# Recreating "update_date" column with the current timestamp
df_old = df_old.withColumn("update_date", current_timestamp())


In [0]:
df_old.printSchema()

In [0]:
display(df_old)

# Preparing df_new

In [0]:
display(df_new)

In [0]:
# Dropping all the columns which are not required
df_new = df_new.drop('old_dim_customer_key','old_customer_id','old_create_date','old_update_date')

# Recreating "update_date" and "create_date" columns with the current timestamp
df_new = df_new.withColumn("create_date", current_timestamp())\
                .withColumn("update_date", current_timestamp())

            
# display(df_old)
# display(df_new)
# df = df_new.union(df_old)
# df = df.withColumn('dim_customer_key', monotonically_increasing_id()+lit(1))
# display(df.limit(20))

In [0]:
# display(df_old)
display(df_new)

<!-- # Surrogate Key - All The Values -->
# Surrogate Key - From 1

In [0]:
# df = df.withColumn('dim_customer_key', monotonically_increasing_id()+lit(1))

# display(df.limit(20))

df_new = df_new.withColumn('dim_customer_key', monotonically_increasing_id()+lit(1))

In [0]:
display(df_new)

# Adding Max Surrogate Key

In [0]:
if init_load_flag == 1:
  max_surrogate_key = 0

else:
  # max_surrogate_key = df_old.select(max('dim_customer_key')).collect()[0][0]
  df_maxsur = spark.sql("SELECT MAX(dim_customer_key) as max_surrogate_key FROM medallion_arc_e2e.gold.dim_customers")

  # Converting df_maxsur to max_surrogate_key varible
  max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']




# df_new = df_new.withColumn('dim_customer_key', col('dim_customer_key')+max_surrogate_key)
# display(df_new)
# df_new

# Converting df_maxsur to max_surrogate_key varible

In [0]:
# df_collect = df.limit(2)

# df_collect.collect()

In [0]:
# max_surrogate_key = df_maxsur.collect()[0]['max_surrogate_key']
# df_new = df_new.withColumn('dim_customer_key', col('dim_customer_key')+max_surrogate_key)

In [0]:
df_new = df_new.withColumn('dim_customer_key', col('dim_customer_key')+ lit(max_surrogate_key))

In [0]:
display(df_new)

# Union of df_old and df_new

In [0]:
df_final = df_new.unionByName(df_old)
# df_final = df_final.withColumn("dim_customer_key", col("dim_customer_key")+max_surrogate_key)
display(df_final)
# df_final.write.mode("overwrite").saveAsTable("medallion_arc_e2e.gold.dim_customers")

# SCD Type - 1

In [0]:
# Instead of flag we can use this condition
if spark.catalog.tableExists("medallion_arc_e2e.gold.dim_customers"):
    print("Table already exists")
else:
    print("Table does not exist")

# Upsert Logic

In [0]:
from delta.tables import DeltaTable
# deltaTable = DeltaTable.forName(spark, "medallion_arc_e2e.gold.dim_customers")
# deltaTable.alias("t").merge(df_final.alias("s"), "t.id = s.id")\
# .whenMatchedUpdateAll()\
# .whenNotMatchedInsertAll()\
# .execute()

In [0]:
if spark.catalog.tableExists("medallion_arc_e2e.gold.dim_customers"):
  # df_final.write.mode("append").saveAsTable("medallion_arc_e2e.gold.dim_customers")
  dlt_obj = DeltaTable.forPath(spark, "abfss://gold-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/dim_customers")
  dlt_obj.alias("trg").merge(df_final.alias("src"), "trg.dim_customer_key = src.dim_customer_key")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
    # .whenMatchedUpdate() # When selected column has to be update then subset = ['col1','col2'] has to pass.
else:
  df_final.write.mode("overwrite")\
                .option("overwriteSchema", "true")\
                .option("path","abfss://gold-e2e-eus-p01@adlsdemodatabrickseus202.dfs.core.windows.net/dim_customers")\
                .saveAsTable("medallion_arc_e2e.gold.dim_customers")

# df_final.write.mode("overwrite").saveAsTable("medallion_arc_e2e.gold.dim_customers")
# df_final.write.mode("append").saveAsTable("medallion_arc_e2e.gold.dim_customers")

In [0]:
%sql
SELECT * FROM medallion_arc_e2e.gold.dim_customers;